In [91]:
import os
import json
import random
import pandas as pd
import numpy as np
import math
import ast
from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize

In [ ]:
df = pd.read_parquet("/content/drive/MyDrive/data_parquet/cleaned2.1_Software.parquet")

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
N_CLUSTERS = 100
SAMPLE_PER_CLUSTER = 8

In [ ]:
model = SentenceTransformer(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def sample_category(df, category_col="category_name", sample_n=1_000_000):
    if len(df) <= sample_n:
        return df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    return df.sample(n=sample_n, random_state=RANDOM_SEED).reset_index(drop=True)

In [ ]:
df_sample = sample_category(df,"software")
df_sample

,parent_asin,review_id,sentence_id,sentence_text,rating
0,B01617VPUY,B01617VPUY,25770020896,five [GENERIC_NOUN] works great and arrived wh...,5.0
1,B00CVBMLOE,B00CVBMLOE,68719665429,the long and short of it is that the buyer sho...,1.0
2,B076M69SR5,B076M69SR5,25770205502,[GENERIC_NOUN] killer pretty good [GENERIC_NOU...,4.0
3,B06XP4F49R,B06XP4F49R,25770048654,motortrend content good inability to skim [GEN...,1.0
4,B0868LWXXJ,B0868LWXXJ,17180257836,game playing granny this game is enjoyable,5.0
...,...,...,...,...,...
999995,B008XG1X18,B008XG1X18,449512,taking screen shots and saving them in a faceb...,1.0
999996,B00CA89UZ6,B00CA89UZ6,68719629642,the camera shook and wobbled and the quality w...,1.0
999997,B018SQEVFM,B018SQEVFM,60129661107,great for toddlers our three [GENERIC_NOUN] ol...,5.0
999998,B008RA3X5E,B008RA3X5E,539984,love the sounds that come with it and the weat...,5.0


In [ ]:
def embed_sentences(sentences, batch_size=256):
    embeddings = model.encode(
        sentences.tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return embeddings

In [ ]:
embeddings = embed_sentences(df_sample['sentence_text'].astype(str))

Batches:   0%|          | 0/3907 [00:00<?, ?it/s]

In [ ]:
def cluster_sentences(embeddings, n_clusters=N_CLUSTERS):
    kmeans = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=RANDOM_SEED,
        batch_size=4096,
        n_init="auto"
    )
    labels = kmeans.fit_predict(embeddings)
    return labels, kmeans

In [ ]:
labels,kmeans = cluster_sentences(embeddings)

In [ ]:
df_sample['cluster_id'] = labels

In [71]:
def pick_8_sentences_per_cluster(df):
    selected_rows = []
    for cluster_id, group in df.groupby("cluster_id"):
        take_n = min(SAMPLE_PER_CLUSTER, len(group))
        chosen = group.sample(n=take_n, random_state=RANDOM_SEED)
        selected_rows.append(chosen)
    if selected_rows:
        return pd.concat(selected_rows, ignore_index=True)
    return df.iloc[0:0].copy()

In [73]:
df_cluster_pick = pick_8_sentences_per_cluster(df_sample)

In [80]:
df_cluster_pick.to_csv("data_gan_nhan.csv", index=False, encoding="utf-8-sig")

In [85]:
df_cluster_pick = pd.read_csv("data_gan_nhan.csv")

In [87]:
df_cluster_pick = df_cluster_pick.drop("parent_asin",axis=1)
df_cluster_pick

,review_id,sentence_id,sentence_text,rating,cluster_id
0,B00LIBW0V2,609669,getting [GENERIC_NOUN] for cars is almost impossible because once you start racing to get upgrade cards you have to race people who have higher powered cars then you and have to race a 5 stage race against 1015 people to get 1 card,4.0,0
1,B00IIY2DP0,25770007138,after getting into it and playing for anywhere between 6 and 11 episodes you get a message about the writer having to submit more of the story in the coming [GENERIC_NOUN],3.0,0
2,B00SSUMT7G,25769895895,[GENERIC_NOUN] game does not show chips draw marker or how to turn off 34double or nothing34 option,5.0,0
3,B009KS4XRO,551966,plus i would like to know how some people get a bingo after only [GENERIC_NOUN] number is called,2.0,0
4,B00ETSKGBM,85899447253,level 11 really liked this game until i got to level eleven but now it wont let me answer the riddle i know the answer but when i choose the first leter of the second word it disappears the other letters go into the first [DOMAIN_NOISE] but not the corrrct [GENERIC_NOUN],1.0,0
...,...,...,...,...,...
795,B00H2GDFZ2,68719532058,great tool for personal use and teaching children easy to use and set budget and spending,5.0,99
796,B009G9KFQ0,17180288311,a excellent has a great interface and simple to use,4.0,99
797,B00NGYBMOK,17179885281,hat finder seems to work well on fire,5.0,99
798,B00EDSI7QO,34359847454,easy to set this up for my wife to use,5.0,99


In [ ]:
OPENAI_API_KEY = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
from openai import OpenAI
import time
_api_key = os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY)
client = OpenAI(api_key=_api_key)
SYSTEM_PROMPT = """You are an ASTE (Aspect-Sentiment-Opinion Triplet Extraction) expert for e-book and Kindle product reviews.

Extract all aspect-opinion-sentiment triplets from the given sentence.

## FORMAT
Return a JSON array of lists: [[aspect, opinion, sentiment], ...]
- aspect: the aspect term (noun/noun phrase being evaluated)
- opinion: the opinion expression (include negation if present, e.g. "not good", "not bad")
- sentiment: 0 (negative), 1 (neutral), 2 (positive)

## RULES
- If no aspect-opinion pair found, return: []
- Keep negation as part of the opinion (e.g. "not good" → sentiment 0, "not bad" → sentiment 2)
- If multiple aspects share the same opinion, create one triplet per aspect
- If one aspect has multiple opinions, create one triplet per opinion
- Implicit opinions (verbs/phrases) are valid: "I love X" → [["X", "love", 2]]
- Sentences with no specific aspect (e.g. "Very good", "I love it", "Absolutely terrible") → []
- Return ONLY the JSON array, no explanation, no markdown

## EXAMPLES
The story is engaging. → [["story", "engaging", 2]]
The ending is predictable. → [["ending", "predictable", 0]]
The story is okay. → [["story", "okay", 1]]
The screen is not good. → [["screen", "not good", 0]]
The keyboard is not bad. → [["keyboard", "not bad", 2]]
The app is not too slow. → [["app", "not too slow", 1]]
I love the keyboard. → [["keyboard", "love", 2]]
I hate the touchpad. → [["touchpad", "hate", 0]]
The app crashes often. → [["app", "crashes often", 0]]
The story is engaging but the ending is disappointing. → [["story", "engaging", 2], ["ending", "disappointing", 0]]
The sound and bass are amazing. → [["sound", "amazing", 2], ["bass", "amazing", 2]]
The plot and characters are boring. → [["plot", "boring", 0], ["characters", "boring", 0]]
The battery is long-lasting and reliable. → [["battery", "long-lasting", 2], ["battery", "reliable", 2]]
The plot is engaging, the writing is smooth, but the ending feels rushed. → [["plot", "engaging", 2], ["writing", "smooth", 2], ["ending", "rushed", 0]]
This book is better than my old one. → [["book", "better", 2]]
Very good. → []
I love it. → []
Absolutely terrible. → []"""

In [89]:
def extract_triplets(sentence: str, max_retries: int = 3) -> list:
    for attempt in range(max_retries):
        try:
            sentence = sentence.replace("[GENERIC_NOUN]", "").replace("[DOMAIN_NOISE]", "").replace("[NEG]", "").strip()
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": sentence},
                ],
                max_tokens=512,
                temperature=0,
            )
            raw = response.choices[0].message.content.strip()
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
            return json.loads(raw)
        except json.JSONDecodeError:
            if attempt == max_retries - 1:
                return []
        except Exception as e:
            err = str(e)
            if "rate_limit" in err.lower() or "429" in err:
                wait = 2 ** (attempt + 2)
                print(f"  Rate limit, waiting {wait}s...")
                time.sleep(wait)
            elif attempt == max_retries - 1:
                print(f"  Error on '{sentence[:40]}...': {e}")
                return []
            else:
                time.sleep(2 ** attempt)
    return []

In [92]:
triplets_all = []
t_start = time.time()

for idx, row in df_cluster_pick.iterrows():
    triplets = extract_triplets(row["sentence_text"])
    triplets_all.append(triplets)

    done = len(triplets_all)
    if done % 100 == 0:
        elapsed = time.time() - t_start
        rate = done / elapsed
        remaining = (len(df_cluster_pick) - done) / rate if rate > 0 else 0
        print(f"  [{done}/{len(df_cluster_pick)}]  {rate:.1f} calls/s  ~{remaining:.0f}s remaining")

df_cluster_pick = df_cluster_pick.copy()
df_cluster_pick["triplets"] = triplets_all
df_cluster_pick["category_name"] = "software"

n_with = sum(1 for t in triplets_all if t)
print(f"\nLabeling done in {time.time() - t_start:.1f}s  —  {n_with}/{len(triplets_all)} sentences with triplets")

  [100/800]  1.8 calls/s  ~388s remaining
  [200/800]  1.8 calls/s  ~337s remaining
  [300/800]  1.8 calls/s  ~278s remaining
  [400/800]  1.8 calls/s  ~222s remaining
  [500/800]  1.8 calls/s  ~165s remaining
  [600/800]  1.8 calls/s  ~112s remaining
  [700/800]  1.8 calls/s  ~56s remaining
  [800/800]  1.8 calls/s  ~0s remaining


PySparkTypeError: [NOT_COLUMN_OR_STR] Argument `col` should be a Column or str, got generator.

In [95]:
df_cluster_pick.to_json("data_labeled.jsonl", orient="records", lines=True, force_ascii=False)

In [103]:
df_cluster_pick.to_csv("data_labeled.csv",index=False)

In [49]:
import ast
df_cluster_pick["triplets"] = df_cluster_pick["triplets"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [50]:
from collections import Counter
import os
import builtins

def generate_report(df, out_path):
    if os.path.isdir(out_path):
        out_path = os.path.join(out_path, "hard_labeling_report_software.txt")

    total_sentences = len(df)
    no_triplet = (df["triplets"].apply(len) == 0).sum()
    multi_triplet = (df["triplets"].apply(len) > 1).sum()
    sentiment_counter = Counter()
    total_triplets = 0
    for triplets in df["triplets"]:
        if isinstance(triplets, (list, tuple)):
            total_triplets += len(triplets)
            for t in triplets:
                if isinstance(t, (list, tuple)) and len(t) > 2:
                    sentiment_counter[t[2]] += 1

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(f"Tổng số câu: {total_sentences}\n")
        f.write(f"Tổng triplets: {total_triplets}\n")
        f.write(f"Số câu không có triplet nào: {no_triplet}\n")
        f.write(f"Số câu có > 1 triplet: {multi_triplet}\n")
        f.write("Sentiment:\n")
        f.write(f"  negative (0): {sentiment_counter.get(0, 0)}\n")
        f.write(f"  neutral  (1): {sentiment_counter.get(1, 0)}\n")
        f.write(f"  positive (2): {sentiment_counter.get(2, 0)}\n")
    print("Saved to:", out_path)

In [100]:
generate_report(df_cluster_pick,'hard_labeling_report_software.txt')

Saved to: hard_labeling_report_software.txt


In [113]:
with open('hard_labeling_report_software.txt', 'r', encoding='utf-8') as f:
    content = f.read()

print(content)

Tổng số câu: 800
Tổng triplets: 806
Số câu không có triplet nào: 379
Số câu có > 1 triplet: 245
Sentiment:
  negative (0): 283
  neutral  (1): 52
  positive (2): 471

